In [16]:
from datasets import load_dataset
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import Dense
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras import Model
from tensorflow.keras.models import load_model

NUM_CLASSES = 101
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

In [2]:
ds = load_dataset("food101", cache_dir="./data")

train_ds = ds["train"].select(range(int(len(ds["train"]) * 0.1)))
test_ds = ds["validation"].select(range(int(len(ds["validation"]) * 0.1)))

In [3]:
def hf_to_tf_dataset(hf_ds, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE):
    def gen():
        for example in hf_ds:
            img = example["image"].convert("RGB").resize(image_size)
            img = np.array(img, dtype=np.float32) / 255.0
            label = tf.keras.utils.to_categorical(example["label"], num_classes=NUM_CLASSES)
            yield img, label
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(*image_size, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(NUM_CLASSES,), dtype=tf.float32)
        )
    )
    return ds.batch(batch_size)

In [4]:
split_idx = int(len(train_ds) * 0.7)
training_set = hf_to_tf_dataset(train_ds.select(range(split_idx)))
validation_set = hf_to_tf_dataset(train_ds.select(range(split_idx, len(train_ds))))
test_set = hf_to_tf_dataset(test_ds)

I0000 00:00:1772610581.915844   14319 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4051 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
pretrained_model = EfficientNetV2B0(weights='imagenet', include_top=False, pooling='avg')

In [6]:
pretrained_model.trainable = False

In [7]:
from tensorflow.keras.layers import Input
inputs = Input(shape=(224,224,3), name='input_layer')

# Tautkan Input Layer pada pre-trained model
revised_model_3 = pretrained_model(inputs)

In [8]:
outputs = Dense(units=101, activation='softmax', name='output_layer')(revised_model_3)

In [9]:
model_3 = Model(inputs, outputs, name='model_3')

In [10]:
model_3.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [11]:
model_3.summary()

Model: "model_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 1280)           │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 101)            │       129,381 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,048,693 (23.07 MB)

 Trainable params: 129,381 (505.39 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [12]:
es_callback = EarlyStopping(monitor='val_loss', patience=5)
mpt_callback= ModelCheckpoint(filepath='best_model_3.keras', save_best_only=True)

In [13]:
model_3.fit(training_set, epochs=100, validation_data = validation_set, callbacks = [es_callback, mpt_callback])

Epoch 1/100


2026-03-04 14:49:47.742232: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f270c052530 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-04 14:49:47.742252: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6
2026-03-04 14:49:47.936916: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


2026-03-04 14:49:48.946248: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


2026-03-04 14:49:49.538502: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


2026-03-04 14:49:50.313885: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11424', 144 bytes spill stores, 144 bytes spill loads



...

164/166 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.2983 - loss: 3.4628

165/166 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.2987 - loss: 3.4605

166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.2990 - loss: 3.4582

166/166 ━━━━━━━━━━━━━━━━━━━━ 24s 147ms/step - accuracy: 0.3510 - loss: 3.0889 - val_accuracy: 0.0000e+00 - val_loss: 12.5679


2026-03-04 14:53:12.248511: I tensorflow/core/framework/local_rendezvous.cc:426] Local rendezvous recv item cancelled. Key hash: 11686127939203887162


In [17]:
best_model_3 = load_model('best_model_3.keras')

In [19]:
best_model_3.evaluate(test_set)

      1/Unknown 5s 5s/step - accuracy: 0.0000e+00 - loss: 7.1133

      3/Unknown 5s 66ms/step - accuracy: 0.0000e+00 - loss: 7.1130

      4/Unknown 5s 78ms/step - accuracy: 0.0000e+00 - loss: 7.1127

      5/Unknown 5s 83ms/step - accuracy: 0.0000e+00 - loss: 7.1125

      6/Unknown 5s 87ms/step - accuracy: 0.0000e+00 - loss: 7.1125

...

2026-03-04 14:57:58.272986: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-04 14:57:58.391097: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


2026-03-04 14:57:59.290571: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-04 14:57:59.405137: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


     79/Unknown 21s 216ms/step - accuracy: 0.0444 - loss: 6.7293

79/79 ━━━━━━━━━━━━━━━━━━━━ 21s 216ms/step - accuracy: 0.0990 - loss: 7.4188


[7.418811321258545, 0.09900990128517151]